# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
from transformers import pipeline

load_dotenv(override=True)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key:
  print(f"openrouter api key found: {openrouter_api_key[:8]}")

MODEL = "openai/gpt-oss-20b:free"
openai = OpenAI(
    base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key
)

system_message = """
    You are a helpful AI assistant.
    Answer all the user queries with short and brief explanation. 
"""


# This function creates any text into a voice (returns raw bytes)
def talker(message):
  response = openai.audio.speech.create(
      model="deepgram/flux-tts:free",
      input=message,
      voice="flux-alexis-en",
      response_format="mp3",
  )
  return response.content

def put_message_in_chatbot(message, history):
  return "", history + [{"role":"user", "content":message}]

def chat(history):
  history = [{"role":h["role"], "content":h["content"]} for h in history]
  messages = [{"role": "system", "content": system_message}] + history

  response = openai.chat.completions.create(
      model=MODEL, messages=messages
  )

  while response.choices[0].finish_reason == "tool_calls":
    message_obj = response.choices[0].message
    responses = handle_tool_calls(message_obj) 
    messages.append(message_obj)
    messages.extend(responses)
    response = openai.chat.completions.create(model=MODEL, messages=messages)

  if response.choices[0].message.content:
    msg = response.choices[0].message.content
    return history, talker(msg)
  return "I encountered an error generating a text response."



# 5. Build the custom interface layout using gr.Blocks
with gr.Blocks() as demo:
  gr.Markdown("### AI Chat Assistant")
  with gr.Row():
    chatbot = gr.Chatbot(type="messages")
  with gr.Row():
    audio_output = gr.Audio(label="AI Voice Response", autoplay=True)
  with gr.Row():
    message = gr.Textbox(label="Chat with our AI Assistant:")

  message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        fn=chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )


if __name__ == "__main__":
  demo.launch()


In [ ]:
import os
import tempfile

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError(
        "OPENROUTER_API_KEY not found. Add it to your .env file before running."
    )
print(f"OpenRouter API key found: {OPENROUTER_API_KEY[:8]}...")

CHAT_MODEL = "openai/gpt-oss-20b:free"
TTS_MODEL = "deepgram/flux-tts:free"
TTS_VOICE = "flux-alexis-en"

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

SYSTEM_MESSAGE = (
    "You are a helpful AI assistant. "
    "Answer all user queries with a short and brief explanation."
)


def talker(text):
    """Synthesize speech and return a file path (gr.Audio cannot take raw bytes)."""
    try:
        response = client.audio.speech.create(
            model=TTS_MODEL,
            input=text,
            voice=TTS_VOICE,
            response_format="mp3",
        )
    except Exception as exc:
        print(f"TTS failed: {exc}")
        return None

    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as handle:
        handle.write(response.content)
        return handle.name


def put_message_in_chatbot(message, history):
    """Clear the textbox and append the user turn to the chat history."""
    history = history or []
    message = (message or "").strip()
    if not message:
        return "", history
    return "", history + [{"role": "user", "content": message}]


def chat(history):
    """Takes the chatbot history, returns (updated history, audio path)."""
    history = history or []
    if not history or history[-1].get("role") != "user":
        return history, None

    print("history:", history)

    messages = [{"role": "system", "content": SYSTEM_MESSAGE}] + [
        {"role": turn["role"], "content": turn["content"]}
        for turn in history
        if turn.get("content")
    ]

    try:
        response = client.chat.completions.create(model=CHAT_MODEL, messages=messages)
    except Exception as exc:
        error_text = f"Sorry, something went wrong: {exc}"
        return history + [{"role": "assistant", "content": error_text}], None

    reply = response.choices[0].message.content
    if not reply:
        reply = "I encountered an error generating a text response."
        return history + [{"role": "assistant", "content": reply}], None

    return history + [{"role": "assistant", "content": reply}], talker(reply)


with gr.Blocks() as demo:
    gr.Markdown("### AI Chat Assistant")
    with gr.Row():
        chatbot = gr.Chatbot(type="messages")
    with gr.Row():
        audio_output = gr.Audio(label="AI Voice Response", type="filepath", autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot],
    ).then(
        fn=chat,
        inputs=chatbot,
        outputs=[chatbot, audio_output],
    )


if __name__ == "__main__":
    demo.launch()